# 00b · Verify DACON Stage 3 metric semantics

Run once before any Stage 3 classification experiment.
This notebook locks the local metric behavior to the competition definition.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import subprocess
import sys

# ============================================================
# Repository
# ============================================================
REPO = Path("/content/Blackbox-Detection")
REPO_URL = "https://github.com/sangchun1/Blackbox-Detection.git"
BRANCH = "stage3-sangchun"

if not REPO.exists():
    subprocess.run(
        [
            "git", "clone",
            "--branch", BRANCH,
            "--single-branch",
            REPO_URL,
            str(REPO),
        ],
        check=True,
    )
else:
    subprocess.run(
        ["git", "-C", str(REPO), "fetch", "origin", BRANCH],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(REPO), "checkout", BRANCH],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(REPO), "pull", "--ff-only", "origin", BRANCH],
        check=True,
    )

# ============================================================
# Paths
# ============================================================
DRIVE_ROOT = Path("/content/drive/MyDrive/Blackbox-Detection")
DATA_ROOT = DRIVE_ROOT / "DATASET"

COMMA_ROOT = DATA_ROOT / "comma2k19"
RAW_ROOT = COMMA_ROOT / "raw"
PROCESSED_ROOT = COMMA_ROOT / "processed" / "v1"

MANIFEST_ROOT = DRIVE_ROOT / "manifests" / "stage3" / "v1"
OUTPUT_ROOT = DRIVE_ROOT / "outputs" / "stage3"
PRETRAINED_ROOT = DRIVE_ROOT / "pretrained"

for p in [PROCESSED_ROOT, MANIFEST_ROOT, OUTPUT_ROOT, PRETRAINED_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

# ============================================================
# Install project from the existing pyproject.toml
# --no-deps keeps Colab's/DACON's binary stack intact.
# ============================================================
subprocess.run(
    [
        sys.executable,
        "-m", "pip", "install",
        "-q", "--no-deps", "-e", str(REPO),
    ],
    check=True,
)

if str(REPO / "src") not in sys.path:
    sys.path.insert(0, str(REPO / "src"))

commit = subprocess.check_output(
    ["git", "-C", str(REPO), "rev-parse", "--short", "HEAD"],
    text=True,
).strip()

print("Repository     :", REPO)
print("Branch         :", BRANCH)
print("Commit         :", commit)
print("RAW_ROOT       :", RAW_ROOT)
print("PROCESSED_ROOT :", PROCESSED_ROOT)

In [ ]:
from blackbox_detection.stage3.metrics import dacon_stage3_metrics

# ============================================================
# Test 1: perfect prediction + GT STOPPED steering exclusion
# ============================================================
truth_a = ["ACCELERATING", "DECELERATING", "CONSTANT", "STOPPED"]
pred_a  = ["ACCELERATING", "DECELERATING", "CONSTANT", "STOPPED"]

truth_s = ["LEFT", "STRAIGHT", "RIGHT", "LEFT"]
pred_s  = ["LEFT", "STRAIGHT", "RIGHT", "RIGHT"]

result = dacon_stage3_metrics(truth_a, pred_a, truth_s, pred_s)
print("perfect / STOPPED exclusion:", result)

assert abs(result["accel_macro_f1"] - 1.0) < 1e-12
assert abs(result["steer_macro_f1"] - 1.0) < 1e-12
assert result["steer_eval_frames"] == 3
assert abs(result["stage3_score"] - 1.0) < 1e-12

In [ ]:
# ============================================================
# Test 2: Macro-F1 must use ALL defined classes
# even if a tiny local subset contains only one class.
# ============================================================
truth_a = ["CONSTANT", "CONSTANT"]
pred_a  = ["CONSTANT", "CONSTANT"]

truth_s = ["STRAIGHT", "STRAIGHT"]
pred_s  = ["STRAIGHT", "STRAIGHT"]

result = dacon_stage3_metrics(truth_a, pred_a, truth_s, pred_s)
print("defined-class macro test:", result)

# accel: CONSTANT F1=1, the other 3 classes=0 -> 1/4
assert abs(result["accel_macro_f1"] - 0.25) < 1e-12

# steer: STRAIGHT F1=1, LEFT/RIGHT=0 -> 1/3
assert abs(result["steer_macro_f1"] - (1.0 / 3.0)) < 1e-12

expected = 0.7 * 0.25 + 0.3 * (1.0 / 3.0)
assert abs(result["stage3_score"] - expected) < 1e-12

In [ ]:
# ============================================================
# Test 3: predicted STOPPED must NOT remove steering rows.
# Only ground-truth STOPPED is excluded.
# ============================================================
truth_a = ["CONSTANT", "STOPPED"]
pred_a  = ["STOPPED", "STOPPED"]

truth_s = ["LEFT", "RIGHT"]
pred_s  = ["RIGHT", "LEFT"]

result = dacon_stage3_metrics(truth_a, pred_a, truth_s, pred_s)
print("GT STOPPED-mask test:", result)

# second row is excluded because GT accel is STOPPED.
# first row is still evaluated although prediction is STOPPED.
assert result["steer_eval_frames"] == 1
assert abs(result["steer_macro_f1"] - 0.0) < 1e-12

print("\nAll DACON Stage 3 metric sanity checks passed.")

**Locked semantics**

- Acceleration: Macro-F1 over all 4 defined classes.
- Steering: Macro-F1 over all 3 defined classes.
- Steering rows are excluded only where **ground-truth acceleration is STOPPED**.
- Stage 3 local score: `0.7 * accel_macro_f1 + 0.3 * steer_macro_f1`.

Keep `src/blackbox_detection/stage3/metrics.py` as the single source of truth for later experiments.